# Reading and Writing Files

Pyspark provides powerful and flexible API to **Read** and **Write** data from a variety of sources including - **CSV**, **JSON**, **Parquet**, **ORC** and **Databases** - using Spark Dataframes interface . These operations from the backbone of most ETL pipelines. enabling the process data at scale in distributed environments.

In this tutorial , we will learn the general pattern for reading and writing files in pyspark. 

## Writing Files in Pyspark

Lets get started with reading files. The general format for reading files in pyspark  is all follows:


```
from pyspark.sql import SparkSession

# Step 1: Initialize SparkSession
spark = SparkSession.builder \
    .appName("Read File Example") \
    .getOrCreate()

# Step 2: Read data
df = spark.read \
    .format("<file_format>") \
    .option("<option_name>", "<option_value>") \
    .load("<path_to_file_or_folder>")

# Step 3: Inspect data
df.show(5)
df.printSchema()


```

### Read Modes Explained

#### Parameters Explained

|Parameter| Description |Example|
|---------|-------------|-------|
|`<file_format>`|Type of data source|csv,json,parquet, orc, jdbc|
|option|Used to configure reading options|.options("header","true")or .option("inferschema","true")|
|load()|path fo file or directory| /datasets/customer.csv|


####  Example 1 : Reading CSV File

In [0]:

df_csv = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/datasets/customers.csv")


#### Example 2: Reading a Parquet file


In [0]:
df_parquet = spark.read \
    .format("parquet") \
    .load("/datasets/sales.parquet")


#### Example 3: Reading a JSON File

In [0]:

df_json = spark.read \
    .format("json") \
    .load("/datasets/events.json")


#### Example 4: Reading from a Hive Table
You can also read data from Hive tables using spark.read.table() or SQL queries.

In [0]:
# Reading a Hive table customers in default schema into a DataFrame
df_hive = spark.read.table("default.customers")

df_hive.show(5)
df_hive.printSchema()

In [0]:
# Querying a Hive table with SQL
df_hive_sql = spark.sql("SELECT id, name, email FROM default.customers WHERE country='US'")
df_hive_sql.show(5)

#### Example 5 : Reading from Database(JDBC)





In [0]:
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/mydb") \
    .option("dbtable", "public.customers") \
    .option("user", "postgres") \
    .option("password", "mypassword") \
    .load()


### Shortcuts for common Formats

For CSV, JSON, and Parquet, you can skip `.format()` and directly use built-in methods. You will see this sytax being used commonly.

In [0]:
df_csv = spark.read.csv(path, header=True, inferSchema=True)
df_json = spark.read.json(path)
df_parquet = spark.read.parquet(path)


### Read Modes Explained
Read modes handle malformed or corrupt records during data loading from formats like JSON, CSV, or Parquet. These modes determine whether to process bad data leniently, skip it, or fail the job.


|Mode|Description|
|----|-----------|
|PERMISSIVE|	Default mode; sets corrupt record fields to null and stores the raw data in _corrupt_record column.|
|DROPMALFORMED|Drops entire rows with any schema mismatch or corruption, loading only valid records.|
|FAILFAST|Throws an exception immediately upon encountering the first malformed record.|

Set these via `.option("mode", "PERMISSIVE")` or `.mode("DROPMALFORMED")` on the DataFrameReader.

Thie format - using `.read.format().options().load()` - is the most universal flexible pattern for reading files in pyspark , especially useful when building reusable code for different data sources.

## Writing Files in Pyspark

This is the genral format for writing files in Pyspark. It works for all file formats.

```
from pyspark.sql import SparkSession

# Step 1: Initialize SparkSession
spark = SparkSession.builder \
    .appName("Write File Example") \
    .getOrCreate()

# Step 2: Write data
df.write \
    .format("<file_format>") \
    .option("<option_name>", "<option_value>") \
    .mode("<save_mode>") \
    .save("<path_to_output_file_or_folder>")

```


### Parameters Explained

|Parameter| Description |Example|
|---------|-------------|-------|
| `<file_format>` |Type of data output|csv,json,parquet, orc, jdbc|
|option() |Used to configure writing options|.options("header","true")or .option("compression","gzip")|
|mode()|Save behaviour when output already exists| "overwrite","append","ignore","error|
|save()|Path to the file or directory | "/output/customer_parquet"|


#### Save modes explained

|Mode|Description|
|----|-----------|
|overwrite|Replace existing data|
|append|Adds data to existing files or tables|
|ignore|Skips writing if the destination exists|
|error or errorifexists|Fails if the destination exists|



### Examples 1: Writing a CSV Files

In [0]:
df_csv.write\
    .format("csv")\
    .option("header","true")\
    .mode("overwrite")\
    .save("/Workspace/Users/anirudh.work.13.12@outlook.com/Pyspark-Works/Data/customer_csv")

### Examples 1: Writing a CSV Files


In [0]:
df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("/output/sales_parquet")


### Example 3 : Writing a JSON File

In [0]:
df.write \
    .format("json") \
    .mode("overwrite") \
    .save("/output/events_json")


#### Example 4: Writing to a Hive Table
You can write a DataFrame to a Hive table using `saveAsTable()`:

In [0]:
# Write DataFrame to a Hive table in the default schema
df.write \
    .mode("overwrite") \
    .saveAsTable("default.customers_copy")


#### Example 5: Writing to a Database (JDBC)


In [0]:
df.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/mydb") \
    .option("dbtable", "public.customers") \
    .option("user", "postgres") \
    .option("password", "mypassword") \
    .mode("append") \
    .save()


As you can see, it follows the same structure as reading files - you just specify the `format`, `options`, and `mode`, then call `.save()`.



### Shortcut for Common Formats
For CSV, JSON, and Parquet, you can skip `.format()` and use built-in write methods directly. This syntax is very common in practice.

In [0]:
df.write.csv(path="/output/customers", header=True, mode="overwrite")
df.write.json(path="/output/events", mode="overwrite")
df.write.parquet(path="/output/sales", mode="overwrite")


This format - using `.write.format().options().mode().save()`- is the most universal and flexible pattern for writing files in PySpark, making it ideal for reusable ETL pipelines and exporting data to multiple destinations.